# SA locale — régions décisionnelles et PCE locaux

Ce notebook teste une analyse de sensibilité locale sur `terrainSA`.

## Objectif

1. Entraîner un `DecisionTreeRegressor` par sortie afin de découper l'espace des paramètres en **N régions locales**.
2. Dans chaque région, entraîner un **petit métamodèle PCE creux** sur les points observés dans cette région.
3. Vérifier la qualité locale par **Q² en validation croisée**.
4. Calculer les indices de Sobol locaux de chaque PCE.
5. Résumer les indices locaux dans **un seul graphique en boxplots**, plutôt que produire N graphiques.

Par défaut, `N_REGIONS = 8`, afin de conserver des régions assez peu nombreuses et mieux peuplées.

In [11]:
# ============================================================
# 1. Imports et configuration
# ============================================================

from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

PROJECT_ROOT = Path("/Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from maelia_sa_pipeline.config import AGRI_FEATURES, FEATURE_LABELS, TARGET_LABELS
from maelia_sa_pipeline.data import load_dataset
from maelia_sa_pipeline.models import (
    _pce_unit_matrix,
    build_preprocessor,
    format_rule,
    prepare_X,
    transformed_feature_names,
)

try:
    import openturns as ot
    OPENTURNS_AVAILABLE = True
    ot.Log.Show(ot.Log.NONE)
except Exception as exc:
    ot = None
    OPENTURNS_AVAILABLE = False
    print("OpenTURNS indisponible : les PCE locaux ne pourront pas être entraînés.", exc)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")

ANALYSIS_DIR = PROJECT_ROOT / "analysis"
LOG_DIR = PROJECT_ROOT / "simulations" / "log_terrainSA"
TARGETS = ["N_lixi", "dCorg", "rdt"]

# Paramètres principaux de l'analyse locale.
N_REGIONS = 8
OUTPUT_DIR = ANALYSIS_DIR / "local_SA_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_REGION_SIZE = 120
TREE_TEST_SIZE = 0.25
CV_FOLDS = 3
LOCAL_Q2_WARNING = 0.50

# PCE local : réduire max_terms ou CV_FOLDS si le calcul devient trop long.
PCE_DEGREE_LOCAL = 2
PCE_MAX_TERMS_LOCAL = 60
PCE_MAX_TRAIN_LOCAL = 2500
RANDOM_SEED = 42

print("Dossier de sortie :", OUTPUT_DIR)

Dossier de sortie : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/local_SA_results


In [12]:
# ============================================================
# 2. Chargement du dataset terrainSA
# ============================================================

bundle = load_dataset(LOG_DIR, targets=TARGETS)
df = bundle.dataframe.copy()
FEATURES = bundle.feature_columns
CATEGORICAL = bundle.categorical_columns
CONTINUOUS = bundle.continuous_columns

print("Dataset :", bundle.dataset_path)
print("Dimensions :", df.shape)
print("Paramètres :", len(FEATURES), "| Catégoriels :", len(CATEGORICAL), "| Continus :", len(CONTINUOUS))
if bundle.warnings:
    print("Avertissements :")
    for warning in bundle.warnings:
        print("-", warning)

display(df.head())

Dataset : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/simulations/log_terrainSA/dataset_metamodel.csv
Dimensions : (10000, 31)
Paramètres : 26 | Catégoriels : 14 | Continus : 12
Avertissements :
- Les colonnes feat_0...feat_25 ont été renommées avec des libellés agronomiques pour l'affichage. Les valeurs restent celles du plan SMT exporté.


,n_ferti,has_prepa,nb_prepa,prepa_1,nb_f1,type_f1_1,nb_f2,type_f2_1,nb_f3,type_f3_1,...,Dose_F1_2,Dose_F2_1,Dose_F2_2,Dose_F3_1,Dose_F3_2,N_lixi,dCorg,rdt,point_idx,parcelle
0,1.0,1.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,55.000000,55.000000,55.000000,55.0,55.0,2.952,-226.304,4.495,0,beauce_5_sa_000
1,2.0,1.0,0.0,0.0,1.0,2.0,1.0,2.0,0.0,0.0,...,42.365041,44.735456,27.061441,55.0,55.0,2.436,-297.342,4.635,1,beauce_5_sa_001
2,0.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,55.000000,55.000000,55.000000,55.0,55.0,2.404,-234.062,4.615,2,beauce_5_sa_002
3,1.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,...,74.751692,55.000000,55.000000,55.0,55.0,2.630,-215.032,4.475,3,beauce_5_sa_003
4,1.0,1.0,1.0,3.0,0.0,2.0,0.0,0.0,0.0,0.0,...,55.000000,55.000000,55.000000,55.0,55.0,2.706,-258.598,4.630,4,beauce_5_sa_004


## Partie I — Découpage local par arbres de décision

On entraîne un arbre par sortie avec `max_leaf_nodes = 8`. Le nombre réel de régions peut être inférieur à 16 si les contraintes `min_samples_leaf` empêchent l'arbre de créer davantage de feuilles stables.

In [13]:
# ============================================================
# 3. Fonctions pour arbres locaux
# ============================================================

def label(name: str) -> str:
    return FEATURE_LABELS.get(name, TARGET_LABELS.get(name, name))


def path_rules_for_leaf(tree: DecisionTreeRegressor, feature_names: list[str], leaf_id: int) -> list[str]:
    children_left = tree.tree_.children_left
    children_right = tree.tree_.children_right
    features_idx = tree.tree_.feature
    thresholds = tree.tree_.threshold
    path: list[str] = []

    def walk(node: int, rules: list[str]) -> bool:
        if node == leaf_id:
            path.extend(rules)
            return True
        left = children_left[node]
        right = children_right[node]
        if left == right:
            return False
        feature_name = feature_names[features_idx[node]]
        threshold = thresholds[node]
        if walk(left, rules + [format_rule(feature_name, threshold, "left", CATEGORICAL)]):
            return True
        if walk(right, rules + [format_rule(feature_name, threshold, "right", CATEGORICAL)]):
            return True
        return False

    walk(0, [])
    return path


def train_region_tree(target: str, seed: int = RANDOM_SEED):
    X = prepare_X(df, FEATURES, CATEGORICAL, CONTINUOUS)
    y = pd.to_numeric(df[target], errors="coerce")
    valid = y.notna()
    X = X.loc[valid]
    y = y.loc[valid]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TREE_TEST_SIZE,
        random_state=seed,
    )
    min_leaf = max(MIN_REGION_SIZE, int(0.01 * len(X_train)))
    pipe = Pipeline([
        ("preprocess", build_preprocessor(CATEGORICAL, CONTINUOUS)),
        ("tree", DecisionTreeRegressor(
            max_leaf_nodes=N_REGIONS,
            min_samples_leaf=min_leaf,
            random_state=seed,
        )),
    ])
    pipe.fit(X_train, y_train)
    pred_train = pipe.predict(X_train)
    pred_test = pipe.predict(X_test)

    metrics = {
        "sortie": target,
        "R2_train": r2_score(y_train, pred_train),
        "Q2_test": r2_score(y_test, pred_test),
        "MAE_test": mean_absolute_error(y_test, pred_test),
        "RMSE_test": mean_squared_error(y_test, pred_test) ** 0.5,
        "n_train": len(X_train),
        "n_test": len(X_test),
        "min_samples_leaf": min_leaf,
    }

    preprocessor = pipe.named_steps["preprocess"]
    tree = pipe.named_steps["tree"]
    feature_names = transformed_feature_names(preprocessor)
    X_all = prepare_X(df, FEATURES, CATEGORICAL, CONTINUOUS)
    leaf_ids = tree.apply(preprocessor.transform(X_all))

    counts = pd.Series(leaf_ids).value_counts().sort_values(ascending=False)
    leaf_to_region = {leaf: f"R{rank:02d}" for rank, leaf in enumerate(counts.index, start=1)}
    regions = []
    global_mean = float(y.mean())
    for leaf, n in counts.items():
        mask = leaf_ids == leaf
        observed = pd.to_numeric(df.loc[mask, target], errors="coerce")
        region = leaf_to_region[leaf]
        regions.append({
            "sortie": target,
            "region": region,
            "leaf_id": int(leaf),
            "n": int(n),
            "part_des_points": float(n / len(df)),
            "moyenne_observee": float(observed.mean()),
            "ecart_a_la_moyenne": float(observed.mean() - global_mean),
            "regles": " ; ".join(path_rules_for_leaf(tree, feature_names, int(leaf))),
        })

    region_series = pd.Series(leaf_ids, index=df.index).map(leaf_to_region)
    return pipe, metrics, pd.DataFrame(regions), region_series

In [14]:
# ============================================================
# 4. Entraînement des arbres et extraction des régions
# ============================================================

tree_models = {}
tree_metric_rows = []
region_tables = []
region_assignments = pd.DataFrame(index=df.index)

for i, target in enumerate(TARGETS):
    print(f"Arbre local — {target}")
    tree_model, tree_metrics, regions, region_series = train_region_tree(target, seed=RANDOM_SEED + i)
    tree_models[target] = tree_model
    tree_metric_rows.append(tree_metrics)
    region_tables.append(regions)
    region_assignments[f"region_{target}"] = region_series
    print(f"  Régions obtenues : {regions['region'].nunique()} / {N_REGIONS}")

regions_df = pd.concat(region_tables, ignore_index=True)
tree_metrics_df = pd.DataFrame(tree_metric_rows)

regions_df.to_csv(OUTPUT_DIR / "decision_tree_regions_locales.csv", index=False)
tree_metrics_df.to_csv(OUTPUT_DIR / "decision_tree_metrics_locales.csv", index=False)
region_assignments.to_csv(OUTPUT_DIR / "region_assignments.csv", index=False)

print("Scores des arbres de régionalisation :")
display(tree_metrics_df.round(3))

print("Aperçu des régions :")
display(regions_df.sort_values(["sortie", "region"]).head(20))

Arbre local — N_lixi
  Régions obtenues : 8 / 8
Arbre local — dCorg
  Régions obtenues : 8 / 8
Arbre local — rdt
  Régions obtenues : 8 / 8
Scores des arbres de régionalisation :


,sortie,R2_train,Q2_test,MAE_test,RMSE_test,n_train,n_test,min_samples_leaf
0,N_lixi,0.768,0.750,0.161,0.216,7500,2500,120
1,dCorg,0.879,0.878,10.115,12.415,7500,2500,120
2,rdt,0.769,0.763,0.048,0.067,7500,2500,120


Aperçu des régions :


,sortie,region,leaf_id,n,part_des_points,moyenne_observee,ecart_a_la_moyenne,regles
0,N_lixi,R01,12,3345,0.3345,3.155899,0.324631,Date_Semis > 74.42 ; Delta_PREPA_Semis > -24.0...
1,N_lixi,R02,10,1322,0.1322,2.361437,-0.469830,Date_Semis ≤ 74.42 ; Date_Recolte ≤ 370.9 ; Da...
2,N_lixi,R03,3,1283,0.1283,2.726006,-0.105261,Date_Semis > 74.42 ; Delta_PREPA_Semis ≤ -24.04
3,N_lixi,R04,14,1115,0.1115,2.743193,-0.088075,Date_Semis ≤ 74.42 ; Date_Recolte ≤ 370.9 ; Da...
4,N_lixi,R05,13,1031,0.1031,2.432023,-0.399244,Date_Semis ≤ 74.42 ; Date_Recolte ≤ 370.9 ; Da...
5,N_lixi,R06,6,996,0.0996,2.881534,0.050267,Date_Semis ≤ 74.42 ; Date_Recolte > 370.9
6,N_lixi,R07,11,631,0.0631,3.500938,0.669671,Date_Semis > 74.42 ; Delta_PREPA_Semis > -24.0...
7,N_lixi,R08,9,277,0.0277,1.775206,-1.056062,Date_Semis ≤ 74.42 ; Date_Recolte ≤ 370.9 ; Da...
8,dCorg,R01,7,1723,0.1723,-211.084670,4.230753,Date_Recolte > 351.7 ; Date_Semis > 68.11 ; Da...
9,dCorg,R02,12,1605,0.1605,-188.828198,26.487224,Date_Recolte ≤ 351.7 ; Date_Semis > 66.52 ; Da...


In [15]:
# ============================================================
# 4bis. Visualisation des régions de l'arbre
# ============================================================

region_plot_df = regions_df.copy()
region_plot_df["sortie_label"] = region_plot_df["sortie"].map(label)
region_plot_df = region_plot_df.sort_values(["sortie", "region"])

fig, axes = plt.subplots(1, len(TARGETS), figsize=(22, 6), sharey=False)
if len(TARGETS) == 1:
    axes = [axes]

for ax, target in zip(axes, TARGETS):
    data = region_plot_df[region_plot_df["sortie"] == target].copy()
    data = data.sort_values("region")
    colors = np.where(data["ecart_a_la_moyenne"] >= 0, "#2A9D8F", "#D7655B")
    bars = ax.bar(data["region"], data["ecart_a_la_moyenne"], color=colors, edgecolor="white", linewidth=1.1)
    ax.axhline(0, color="#263238", linewidth=1)
    ax.set_title(label(target))
    ax.set_xlabel("Région")
    ax.set_ylabel("Écart à la moyenne globale")
    ax.tick_params(axis="x", rotation=45)
    for bar, n, part in zip(bars, data["n"], data["part_des_points"]):
        y = bar.get_height()
        va = "bottom" if y >= 0 else "top"
        offset = 0.015 * max(1e-9, abs(data["ecart_a_la_moyenne"]).max())
        ax.text(bar.get_x() + bar.get_width() / 2, y + (offset if y >= 0 else -offset), f"n={n}\n{part:.0%}",
                ha="center", va=va, fontsize=8, color="#34434A")

fig.suptitle("Régions définies par les arbres : taille et écart moyen", y=1.04, fontsize=20, fontweight="bold")
fig.text(
    0.01, -0.03,
    "Lecture : chaque barre est une région locale; la hauteur indique si la sortie moyenne y est supérieure ou inférieure à la moyenne globale.",
    fontsize=11, color="#6B7280",
)
plt.tight_layout()
region_viz_path = OUTPUT_DIR / "region_tree_overview.png"
fig.savefig(region_viz_path, dpi=190, bbox_inches="tight", facecolor="white")
plt.show()

print("Figure écrite :", region_viz_path)


Figure écrite : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/local_SA_results/region_tree_overview.png


## Partie II — PCE locaux par région

Pour chaque sortie et chaque région de l'arbre, on entraîne un PCE local si la région contient suffisamment de points. La qualité locale est évaluée par un Q² en validation croisée.

Une région est considérée fragile si :

- elle contient moins de `MIN_REGION_SIZE` points ;
- la variance de la sortie est quasi nulle ;
- le PCE échoue ;
- le Q² local est faible.

In [16]:
# ============================================================
# 5. Fonctions PCE local + validation croisée
# ============================================================

if not OPENTURNS_AVAILABLE:
    raise RuntimeError("OpenTURNS est requis pour cette partie.")


def build_sparse_pce(X_np: np.ndarray, y_np: np.ndarray, features: list[str], degree: int, max_terms: int):
    dim = X_np.shape[1]
    distribution = ot.JointDistribution([ot.Uniform(0.0, 1.0)] * dim)
    distribution.setDescription(features)
    poly_factories = [ot.StandardDistributionPolynomialFactory(distribution.getMarginal(i)) for i in range(dim)]
    enumerate_function = ot.LinearEnumerateFunction(dim)
    basis = ot.OrthogonalProductPolynomialFactory(poly_factories, enumerate_function)
    n_total = enumerate_function.getStrataCumulatedCardinal(degree)
    n_keep = min(max_terms, n_total)
    adaptive_strategy = ot.CleaningStrategy(basis, n_total, n_keep, 1e-6)
    projection_strategy = ot.LeastSquaresStrategy()
    algo = ot.FunctionalChaosAlgorithm(
        ot.Sample(X_np),
        ot.Sample(y_np.reshape(-1, 1)),
        distribution,
        adaptive_strategy,
        projection_strategy,
    )
    algo.run()
    result = algo.getResult()
    return result, result.getMetaModel()


def predict_pce(metamodel, X_np: np.ndarray) -> np.ndarray:
    return np.array(metamodel(ot.Sample(X_np))).ravel()


def local_pce_for_region(region_df: pd.DataFrame, target: str, seed: int):
    n_region = len(region_df)
    if n_region < MIN_REGION_SIZE:
        return None, {
            "status": "skipped_small_region",
            "n": n_region,
            "Q2_cv": np.nan,
            "R2_full": np.nan,
            "message": f"Région trop petite (< {MIN_REGION_SIZE})",
        }

    X_unit = _pce_unit_matrix(region_df, FEATURES, CATEGORICAL, CONTINUOUS)
    y = pd.to_numeric(region_df[target], errors="coerce")
    valid = y.notna() & X_unit.notna().all(axis=1)
    X_valid = X_unit.loc[valid].reset_index(drop=True)
    y_valid = y.loc[valid].reset_index(drop=True)

    if len(y_valid) < MIN_REGION_SIZE:
        return None, {
            "status": "skipped_not_enough_valid_points",
            "n": int(len(y_valid)),
            "Q2_cv": np.nan,
            "R2_full": np.nan,
            "message": "Pas assez de points valides",
        }
    if float(y_valid.var(ddof=1)) <= 1e-12:
        return None, {
            "status": "skipped_near_constant_output",
            "n": int(len(y_valid)),
            "Q2_cv": np.nan,
            "R2_full": np.nan,
            "message": "Variance locale quasi nulle",
        }

    n_splits = min(CV_FOLDS, max(2, len(y_valid) // MIN_REGION_SIZE))
    if n_splits < 2:
        n_splits = 2
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_pred = np.full(len(y_valid), np.nan)
    fold_errors = []

    for fold, (train_idx, test_idx) in enumerate(kfold.split(X_valid), start=1):
        X_train = X_valid.iloc[train_idx]
        y_train = y_valid.iloc[train_idx]
        X_test = X_valid.iloc[test_idx]
        try:
            result, metamodel = build_sparse_pce(
                X_train.to_numpy(),
                y_train.to_numpy(dtype=float),
                FEATURES,
                PCE_DEGREE_LOCAL,
                PCE_MAX_TERMS_LOCAL,
            )
            oof_pred[test_idx] = predict_pce(metamodel, X_test.to_numpy())
        except Exception as exc:
            fold_errors.append(f"fold {fold}: {exc}")

    ok_pred = np.isfinite(oof_pred)
    if ok_pred.sum() < max(10, int(0.5 * len(y_valid))):
        return None, {
            "status": "error_cv_failed",
            "n": int(len(y_valid)),
            "Q2_cv": np.nan,
            "R2_full": np.nan,
            "message": " ; ".join(fold_errors[:3]),
        }

    q2_cv = r2_score(y_valid.loc[ok_pred], oof_pred[ok_pred])

    X_final = X_valid
    y_final = y_valid
    if len(X_final) > PCE_MAX_TRAIN_LOCAL:
        X_final = X_final.sample(PCE_MAX_TRAIN_LOCAL, random_state=seed)
        y_final = y_final.loc[X_final.index]

    try:
        result, metamodel = build_sparse_pce(
            X_final.to_numpy(),
            y_final.to_numpy(dtype=float),
            FEATURES,
            PCE_DEGREE_LOCAL,
            PCE_MAX_TERMS_LOCAL,
        )
        pred_full = predict_pce(metamodel, X_final.to_numpy())
        sensitivity = ot.FunctionalChaosSobolIndices(result)
    except Exception as exc:
        return None, {
            "status": "error_final_fit_failed",
            "n": int(len(y_valid)),
            "Q2_cv": float(q2_cv),
            "R2_full": np.nan,
            "message": str(exc),
        }

    sobol_rows = []
    for j, feature in enumerate(FEATURES):
        sobol_rows.append({
            "parametre": feature,
            "Sobol_S1": max(0.0, float(sensitivity.getSobolIndex(j, 0))),
            "Sobol_ST": max(0.0, float(sensitivity.getSobolTotalIndex(j, 0))),
        })

    metrics = {
        "status": "ok" if q2_cv >= LOCAL_Q2_WARNING else "weak_q2",
        "n": int(len(y_valid)),
        "Q2_cv": float(q2_cv),
        "R2_full": float(r2_score(y_final, pred_full)),
        "MAE_cv": float(mean_absolute_error(y_valid.loc[ok_pred], oof_pred[ok_pred])),
        "RMSE_cv": float(mean_squared_error(y_valid.loc[ok_pred], oof_pred[ok_pred]) ** 0.5),
        "cv_folds": int(n_splits),
        "degree": int(PCE_DEGREE_LOCAL),
        "max_terms": int(PCE_MAX_TERMS_LOCAL),
        "n_terms_effectifs": int(result.getIndices().getSize()),
        "message": "" if not fold_errors else " ; ".join(fold_errors[:3]),
    }
    return pd.DataFrame(sobol_rows), metrics

In [17]:
# ============================================================
# 6. Exécution des PCE locaux
# ============================================================

local_metric_rows = []
local_sobol_rows = []

for target_index, target in enumerate(TARGETS):
    region_col = f"region_{target}"
    for region in sorted(region_assignments[region_col].dropna().unique()):
        mask = region_assignments[region_col] == region
        region_df = df.loc[mask].copy()
        print(f"PCE local — {target} / {region} / n={len(region_df)}")
        sobol_part, metrics = local_pce_for_region(
            region_df,
            target,
            seed=RANDOM_SEED + 100 * target_index + int(region.replace("R", "")),
        )
        metric_row = {
            "sortie": target,
            "region": region,
            "part_des_points": float(mask.mean()),
            **metrics,
        }
        local_metric_rows.append(metric_row)
        if sobol_part is not None:
            sobol_part = sobol_part.copy()
            sobol_part["sortie"] = target
            sobol_part["region"] = region
            sobol_part["n_region"] = metrics["n"]
            sobol_part["Q2_cv"] = metrics["Q2_cv"]
            sobol_part["status"] = metrics["status"]
            sobol_part["part_des_points"] = float(mask.mean())
            local_sobol_rows.append(sobol_part)

local_metrics_df = pd.DataFrame(local_metric_rows)
local_sobol_all_df = pd.concat(local_sobol_rows, ignore_index=True) if local_sobol_rows else pd.DataFrame()

accepted_region_keys = local_metrics_df.loc[
    (local_metrics_df["status"] == "ok")
    & (pd.to_numeric(local_metrics_df["Q2_cv"], errors="coerce") >= LOCAL_Q2_WARNING),
    ["sortie", "region"],
]

if local_sobol_all_df.empty:
    local_sobol_df = local_sobol_all_df.copy()
else:
    local_sobol_df = local_sobol_all_df.merge(
        accepted_region_keys.assign(region_accepted=True),
        on=["sortie", "region"],
        how="left",
    )
    local_sobol_df = local_sobol_df[local_sobol_df["region_accepted"].eq(True)].drop(columns="region_accepted")

excluded_regions_df = local_metrics_df.merge(
    accepted_region_keys.assign(region_accepted=True),
    on=["sortie", "region"],
    how="left",
)
excluded_regions_df = excluded_regions_df[~excluded_regions_df["region_accepted"].eq(True)].drop(columns="region_accepted")

local_metrics_df.to_csv(OUTPUT_DIR / "local_pce_metrics.csv", index=False)
local_sobol_all_df.to_csv(OUTPUT_DIR / "local_pce_sobol_indices_all_regions.csv", index=False)
local_sobol_df.to_csv(OUTPUT_DIR / "local_pce_sobol_indices.csv", index=False)
excluded_regions_df.to_csv(OUTPUT_DIR / "local_pce_excluded_regions.csv", index=False)

print("Bilan PCE locaux :")
display(local_metrics_df.groupby(["sortie", "status"]).size().reset_index(name="n_regions"))
print(f"Régions conservées pour la suite : {len(accepted_region_keys)} / {len(local_metrics_df)}")
print(f"Régions exclues pour Q² insuffisant ou échec : {len(excluded_regions_df)}")
display(local_metrics_df.round(3))


PCE local — N_lixi / R01 / n=3345
PCE local — N_lixi / R02 / n=1322
PCE local — N_lixi / R03 / n=1283
PCE local — N_lixi / R04 / n=1115
PCE local — N_lixi / R05 / n=1031
PCE local — N_lixi / R06 / n=996
PCE local — N_lixi / R07 / n=631
PCE local — N_lixi / R08 / n=277
PCE local — dCorg / R01 / n=1723
PCE local — dCorg / R02 / n=1605
PCE local — dCorg / R03 / n=1555
PCE local — dCorg / R04 / n=1480
PCE local — dCorg / R05 / n=1083
PCE local — dCorg / R06 / n=954
PCE local — dCorg / R07 / n=812
PCE local — dCorg / R08 / n=788
PCE local — rdt / R01 / n=4152
PCE local — rdt / R02 / n=2029
PCE local — rdt / R03 / n=1602
PCE local — rdt / R04 / n=694
PCE local — rdt / R05 / n=664
PCE local — rdt / R06 / n=401
PCE local — rdt / R07 / n=281
PCE local — rdt / R08 / n=177
Bilan PCE locaux :


,sortie,status,n_regions
0,N_lixi,ok,7
1,N_lixi,weak_q2,1
2,dCorg,ok,8
3,rdt,ok,7
4,rdt,weak_q2,1


Régions conservées pour la suite : 22 / 24
Régions exclues pour Q² insuffisant ou échec : 2


,sortie,region,part_des_points,status,n,Q2_cv,R2_full,MAE_cv,RMSE_cv,cv_folds,degree,max_terms,n_terms_effectifs,message
0,N_lixi,R01,0.334,ok,3345,0.916,0.926,0.040,0.051,3,2,60,61,
1,N_lixi,R02,0.132,ok,1322,0.875,0.903,0.047,0.066,3,2,60,61,
2,N_lixi,R03,0.128,ok,1283,0.859,0.891,0.070,0.087,3,2,60,61,
3,N_lixi,R04,0.112,ok,1115,0.879,0.884,0.031,0.041,3,2,60,61,
4,N_lixi,R05,0.103,ok,1031,0.661,0.740,0.084,0.110,3,2,60,61,
5,N_lixi,R06,0.100,ok,996,0.631,0.738,0.143,0.197,3,2,60,61,
6,N_lixi,R07,0.063,ok,631,0.957,0.974,0.043,0.055,3,2,60,61,
7,N_lixi,R08,0.028,weak_q2,277,0.498,0.928,0.136,0.193,2,2,60,61,
8,dCorg,R01,0.172,ok,1723,0.996,0.997,0.618,0.772,3,2,60,61,
9,dCorg,R02,0.160,ok,1605,0.996,0.997,0.616,0.770,3,2,60,61,


In [18]:
# ============================================================
# 6bis. Visualisation de la qualité des PCE locaux
# ============================================================

q2_plot_df = local_metrics_df.copy()
q2_plot_df["sortie_label"] = q2_plot_df["sortie"].map(label)
q2_plot_df["Q2_cv_plot"] = pd.to_numeric(q2_plot_df["Q2_cv"], errors="coerce")

fig, axes = plt.subplots(1, len(TARGETS), figsize=(22, 5.8), sharey=True)
if len(TARGETS) == 1:
    axes = [axes]

for ax, target in zip(axes, TARGETS):
    data = q2_plot_df[q2_plot_df["sortie"] == target].sort_values("region")
    colors = data["status"].map({"ok": "#2A9D8F", "weak_q2": "#E9A03F"}).fillna("#D7655B")
    ax.bar(data["region"], data["Q2_cv_plot"], color=colors, edgecolor="white", linewidth=1.1)
    ax.axhline(LOCAL_Q2_WARNING, color="#263238", linestyle="--", linewidth=1.2, label=f"seuil {LOCAL_Q2_WARNING:.2f}")
    ax.set_title(label(target))
    ax.set_xlabel("Région")
    ax.set_ylabel("Q² local en validation croisée")
    ax.set_ylim(min(-0.1, np.nanmin(q2_plot_df["Q2_cv_plot"]) - 0.05), 1.02)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(frameon=False, fontsize=9)

fig.suptitle("Qualité des PCE locaux par région", y=1.04, fontsize=20, fontweight="bold")
fig.text(
    0.01, -0.03,
    "Lecture : les régions orange ou rouges sont exclues des boxplots et des synthèses Sobol locales.",
    fontsize=11, color="#6B7280",
)
plt.tight_layout()
q2_viz_path = OUTPUT_DIR / "local_pce_Q2_by_region.png"
fig.savefig(q2_viz_path, dpi=190, bbox_inches="tight", facecolor="white")
plt.show()

print("Figure écrite :", q2_viz_path)


Figure écrite : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/local_SA_results/local_pce_Q2_by_region.png


## Partie III — Synthèse des indices locaux

Les régions dont le PCE local est trop faible sont exclues avant de produire les synthèses graphiques et tabulaires. Le seuil `LOCAL_Q2_WARNING` est volontairement visible et modifiable.

Le graphique final montre la distribution des indices `ST` locaux entre régions. Il répond à la question : **un paramètre est-il important partout, ou seulement dans certains régimes ?**

In [19]:
# ============================================================
# 7. Boxplots des indices ST locaux
# ============================================================

if local_sobol_df.empty:
    raise RuntimeError(
        "Aucun indice Sobol local disponible après exclusion des régions à Q² insuffisant. "
        "Réduire N_REGIONS, abaisser LOCAL_Q2_WARNING, ou simplifier le PCE local."
    )

plot_df = local_sobol_df.copy()

# On garde les paramètres les plus souvent importants pour éviter un graphique illisible.
top_params = (
    plot_df.groupby("parametre")["Sobol_ST"]
    .median()
    .sort_values(ascending=False)
    .head(14)
    .index
    .tolist()
)
plot_df = plot_df[plot_df["parametre"].isin(top_params)].copy()
plot_df["parametre_label"] = plot_df["parametre"].map(label)
plot_df["sortie_label"] = plot_df["sortie"].map(label)

order = (
    plot_df.groupby("parametre_label")["Sobol_ST"]
    .median()
    .sort_values(ascending=True)
    .index
    .tolist()
)

fig, axes = plt.subplots(1, len(TARGETS), figsize=(22, max(7, 0.45 * len(order) + 2)), sharey=True)
if len(TARGETS) == 1:
    axes = [axes]

for ax, target in zip(axes, TARGETS):
    data = plot_df[plot_df["sortie"] == target]
    sns.boxplot(
        data=data,
        x="Sobol_ST",
        y="parametre_label",
        order=order,
        color="#b9ddd7",
        fliersize=2,
        linewidth=1.1,
        ax=ax,
    )
    sns.stripplot(
        data=data,
        x="Sobol_ST",
        y="parametre_label",
        order=order,
        color="#263238",
        alpha=0.42,
        size=4,
        jitter=0.18,
        ax=ax,
    )
    ax.set_title(label(target))
    ax.set_xlabel("Indice Sobol total local (ST)")
    ax.set_ylabel("")
    ax.set_xlim(left=0)

fig.suptitle("Dispersion des influences locales entre régions — PCE locaux", y=1.02, fontsize=20, fontweight="bold")
fig.text(
    0.01,
    -0.02,
    f"Lecture : chaque point est une région conservée après filtrage Q² local ≥ {LOCAL_Q2_WARNING:.2f}; les régions faibles sont exclues.",
    fontsize=11,
    color="#6B7280",
)
plt.tight_layout()
boxplot_path = OUTPUT_DIR / "local_pce_sobol_ST_boxplots.png"
fig.savefig(boxplot_path, dpi=190, bbox_inches="tight", facecolor="white")
plt.show()

print("Figure écrite :", boxplot_path)


Figure écrite : /Users/benjamin/files/Repositories/Sensitivity_analysis_MAELIA/analysis/local_SA_results/local_pce_sobol_ST_boxplots.png


In [20]:
# ============================================================
# 8. Aide à l'interprétation : régions et paramètres instables
# ============================================================

if local_sobol_df.empty:
    raise RuntimeError("Aucune région n'a été conservée après filtrage Q²; aucune synthèse Sobol locale fiable ne peut être calculée.")

# local_sobol_df contient uniquement les régions conservées après filtrage Q².
summary = (
    local_sobol_df
    .groupby(["sortie", "parametre"], as_index=False)
    .agg(
        median_ST=("Sobol_ST", "median"),
        q25_ST=("Sobol_ST", lambda s: s.quantile(0.25)),
        q75_ST=("Sobol_ST", lambda s: s.quantile(0.75)),
        max_ST=("Sobol_ST", "max"),
        n_regions=("region", "nunique"),
        median_Q2=("Q2_cv", "median"),
    )
)
summary["IQR_ST"] = summary["q75_ST"] - summary["q25_ST"]
summary = summary.sort_values(["sortie", "median_ST"], ascending=[True, False])
summary.to_csv(OUTPUT_DIR / "local_pce_sobol_summary.csv", index=False)

print("Top paramètres par médiane ST locale, régions faibles exclues :")
display(summary.groupby("sortie").head(8).round(3))

print("Paramètres les plus variables entre régions conservées :")
display(summary.sort_values(["sortie", "IQR_ST"], ascending=[True, False]).groupby("sortie").head(8).round(3))

print("Régions exclues de la synthèse finale :")
display(excluded_regions_df.sort_values(["sortie", "region"]).round(3))


Top paramètres par médiane ST locale, régions faibles exclues :


,sortie,parametre,median_ST,q25_ST,q75_ST,max_ST,n_regions,median_Q2,IQR_ST
12,N_lixi,has_prepa,0.182,0.131,0.249,0.477,7,0.875,0.117
16,N_lixi,nb_f3,0.144,0.017,0.165,0.210,7,0.875,0.148
15,N_lixi,nb_f2,0.110,0.106,0.171,0.747,7,0.875,0.065
17,N_lixi,nb_prepa,0.109,0.012,0.155,0.168,7,0.875,0.143
14,N_lixi,nb_f1,0.106,0.066,0.117,0.465,7,0.875,0.051
25,N_lixi,type_f3_2,0.056,0.007,0.061,0.103,7,0.875,0.054
13,N_lixi,n_ferti,0.054,0.011,0.070,0.143,7,0.875,0.059
19,N_lixi,prepa_2,0.054,0.002,0.060,0.084,7,0.875,0.057
38,dCorg,has_prepa,0.188,0.185,0.194,0.202,8,0.994,0.009
42,dCorg,nb_f3,0.181,0.177,0.187,0.206,8,0.994,0.010


Paramètres les plus variables entre régions conservées :


,sortie,parametre,median_ST,q25_ST,q75_ST,max_ST,n_regions,median_Q2,IQR_ST
9,N_lixi,Dose_F2_2,0.005,0.001,0.429,0.996,7,0.875,0.428
16,N_lixi,nb_f3,0.144,0.017,0.165,0.210,7,0.875,0.148
17,N_lixi,nb_prepa,0.109,0.012,0.155,0.168,7,0.875,0.143
12,N_lixi,has_prepa,0.182,0.131,0.249,0.477,7,0.875,0.117
3,N_lixi,Date_Recolte,0.019,0.002,0.080,0.160,7,0.875,0.079
4,N_lixi,Date_Semis,0.021,0.002,0.070,0.134,7,0.875,0.067
15,N_lixi,nb_f2,0.110,0.106,0.171,0.747,7,0.875,0.065
13,N_lixi,n_ferti,0.054,0.011,0.070,0.143,7,0.875,0.059
47,dCorg,type_f1_2,0.008,0.000,0.017,0.020,8,0.994,0.017
40,dCorg,nb_f1,0.127,0.118,0.134,0.153,8,0.994,0.017


Régions exclues de la synthèse finale :


,sortie,region,part_des_points,status,n,Q2_cv,R2_full,MAE_cv,RMSE_cv,cv_folds,degree,max_terms,n_terms_effectifs,message
7,N_lixi,R08,0.028,weak_q2,277,0.498,0.928,0.136,0.193,2,2,60,61,
23,rdt,R08,0.018,weak_q2,177,-1.576,0.978,0.086,0.167,2,2,60,61,


## Lecture attendue

- Si un paramètre a une médiane `ST` élevée et une boîte courte, il est influent dans la plupart des régions.
- Si un paramètre a une boîte très étalée, il est **régime-dépendant** : important dans certains sous-espaces, faible ailleurs.
- Si beaucoup de régions ont un Q² local faible, elles sont exclues de la synthèse finale. Si cela retire trop de régions, l'approche locale n'est pas fiable telle quelle : il faut augmenter le nombre de points, réduire `N_REGIONS`, simplifier le PCE, ou préférer une autre forme de métamodèle local.
- Si l'arbre de régionalisation a un Q² faible, les régions elles-mêmes ne doivent pas être sur-interprétées.